# Berry–Esseen: b4-q3 — CPU computation, logs and checkpoints

Select **Runtime → Run all**. This notebook is preconfigured for the selected task.
Google may request Drive authorization so that checkpoints survive a disconnected session.
Keep the code unchanged and do not run concurrent copies of the same task.

A short benchmark precedes up to 3 hours of allocated search, divided into bounded slices.
Each slice has an external wall-clock limit. A complete candidate forest is submitted to a separate native checker.
Progress is displayed in the cell and recorded in files. Partial results retain both checkpoint copies.
This notebook concerns one conditional n25 count forest; it does not establish the all-index theorem.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
CODE_COMMIT = "651bdb2efc556121e08624001fb8788d0b09ae9d"
TASK = "b4-q3"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Fetching the pinned source commit:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Source commit mismatch")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
# A rerun in an existing Colab kernel must not reuse modules from an older pin.
for module_name in tuple(sys.modules):
    if module_name == "be_replay" or module_name.startswith("be_replay."):
        del sys.modules[module_name]
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
if not Path(sys.modules["be_replay"].__file__).resolve().is_relative_to(CODE.resolve()):
    raise RuntimeError("Imported execution code does not belong to the pinned checkout")
print(json.dumps(snapshot(), indent=2), flush=True)
print("Source code and environment recorded. Mathematical acceptance has not yet been established.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logs and checkpoints:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Testing the supervisor on the actual Linux runtime. These lightweight fixtures do not verify the mathematics.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Supervisor self-tests failed; computation will not start. Log: " + str(SELFTEST))
print("Supervisor self-tests passed. Starting the selected calculation.", flush=True)


In [ ]:
from be_replay.colab_batch import run_count
RESULT = run_count("b4-q3", durable_root=DURABLE,
                   total_seconds=10800, slice_seconds=1800, backup_interval=60)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Execution status:", RESULT["status"]["state"], flush=True)
print("The global proof remains unverified.", flush=True)
print("Results ZIP on Google Drive:", RESULT["bundle"], flush=True)
display(FileLink(RESULT["bundle"]))
try:
    files.download(RESULT["bundle"])
except Exception as error:
    print("Download the ZIP from the Google Drive path shown above:", error, flush=True)


In [ ]:
# FINAL CELL: flush saved Drive results and release the Colab runtime.
# Runs after the final results ZIP is saved, including WAITING/INCOMPLETE.
# Do not run this cell manually while calculations or result writes are active.
# A synchronization error stops this cell BEFORE the runtime is released.
import os as _shutdown_os
import sys as _shutdown_sys

_managed_colab = (globals().get("BERRY_ESSEEN_DRIVE_MOUNTED") is True
                  or bool(_shutdown_os.environ.get("TBE_RUNTIME_ADDR")))
if not _managed_colab:
    print("Outside a managed Colab runtime; skipping shutdown.", flush=True)
else:
    from google.colab import drive as _shutdown_drive
    from google.colab import runtime as _shutdown_runtime

    print("Flushing saved results to Google Drive...", flush=True)
    _shutdown_sys.stdout.flush()
    _shutdown_sys.stderr.flush()
    _shutdown_os.sync()
    _shutdown_drive.flush_and_unmount(timeout_ms=300_000)
    print("Drive synchronization finished. Releasing the Colab runtime.", flush=True)
    _shutdown_runtime.unassign()
